# Report for practical lesson 02: Convolutional Neural Network

In [1]:
import pandas as pd
import os, sys
import numpy as np
project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root_path not in sys.path:
    sys.path.append(project_root_path)

from sklearn.model_selection import train_test_split
import torch

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

This box imports customized modules

In [2]:
from src.data.load import load_data
from src.train_model import train_model

Crucial variables

In [3]:
DATA_PATH = r"c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-02\dataset"

## Data Preprocessing

### Work 01: Load dataset

In [4]:
original_train, original_val, original_test, classes = load_data(
    data_root=DATA_PATH,
    num_workers=0, pin_memory=False,
    batch_size=32
)

### Work 02: Turn dataset into tensor format

In [5]:
def create_tensor_loaders(train_loader, val_loader, test_loader):
    """
    Create tensor transforms and corresponding DataLoaders for train, validation, and test datasets.

    Args:
        train_loader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        val_loader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        test_loader (torch.utils.data.DataLoader): DataLoader for the test dataset.

    Returns:
        tuple: (train_tensor_loader, val_tensor_loader, test_tensor_loader)
            - train_tensor_loader: DataLoader with tensor transform applied to training data.
            - val_tensor_loader: DataLoader with tensor transform applied to validation data.
            - test_tensor_loader: DataLoader with tensor transform applied to test data.
    """

    train_tensor_loader = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )

    val_tensor_loader = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    test_tensor_loader = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    return train_tensor_loader, val_tensor_loader, test_tensor_loader

In [6]:
train_tensor_loader, val_tensor_loader, test_tensor_loader = create_tensor_loaders(
    original_train, original_val, original_test
)

images, labels = next(iter(train_tensor_loader))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


### Work 02: Transform data into grayscale

In [7]:
def to_grayscale(train_loader, val_loader, test_loader):
    """
    Convert RGB images to grayscale for each data loader
    
    Args:
        train_loader: DataLoader with RGB images
        val_loader: DataLoader with RGB images
        test_loader: DataLoader with RGB images
        
    Returns:
        Three DataLoaders containing grayscale images
    """
    grayscale_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1)
    ])
    
    # Create new dataloaders with grayscale transform
    train_gray = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )
    
    val_gray = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_gray = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    return train_gray, val_gray, test_gray

In [8]:
train, val, test = to_grayscale(
    train_tensor_loader, 
    val_tensor_loader, 
    test_tensor_loader
)

In [9]:
images, labels = next(iter(train))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


## Session 01

### Worrk 01: Build Neural Network

The current model was establish via Python code (`lesson-02\networks\model01.py`). The bellow code snippet is to load model into this notebook.

In [10]:
from src.networks.model01 import model01

In [11]:
print("Number of classes:", len(classes))
instance01 = model01(num_classes=len(classes))
print("Instance 01 created successfully.")

Number of classes: 21
Instance 01 created successfully.


To print model description, run this.

In [12]:
print(instance01)

model01(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=21, bias=True)
)


### Work 02: Train model

In [13]:
from src.train_model import train_model

In [14]:
def preprocessing_fn(batch: torch.Tensor) -> torch.Tensor:
    # Convert from [B, 3, 224, 224] -> [B, 1, 28, 28]
    gray = batch.mean(dim=1, keepdim=True)  # RGB to grayscale
    resized = torch.nn.functional.interpolate(gray, size=(28, 28), mode='bilinear', align_corners=False)
    return resized

In [15]:
dev = print('cuda' if torch.cuda.is_available() else 'cpu')

cuda


In [16]:
trained_model, history, best_ckpt_path, test_metrics = train_model(
    train_loader=train,
    val_loader=val,
    model=instance01,  
    epochs=10,
    lr=0.005,
    preprocessing_fn=preprocessing_fn,
    device=dev,
    model_args=None 
)

Epoch 1/10 - train:  71%|███████   | 200/283 [03:40<01:35,  1.16s/batch, loss=2.94]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/10 - val: 100%|██████████| 32/32 [00:34<00:00,  1.07s/batch, val_loss=2.91]


Epoch 1: train_loss=2.9309 val_loss=2.9071 acc=0.1215 prec=0.0235 rec=0.0777 f1=0.0359


Epoch 2/10 - train:  68%|██████▊   | 192/283 [03:39<01:44,  1.15s/batch, loss=2.89]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/10 - val: 100%|██████████| 32/32 [00:13<00:00,  2.37batch/s, val_loss=2.86]


Epoch 2: train_loss=2.8889 val_loss=2.8597 acc=0.1365 prec=0.0280 rec=0.0865 f1=0.0402


Epoch 3/10 - train:  88%|████████▊ | 249/283 [02:01<00:13,  2.43batch/s, loss=2.87]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 3/10 - val: 100%|██████████| 32/32 [00:24<00:00,  1.32batch/s, val_loss=2.88]


...

Epoch 8: train_loss=2.7780 val_loss=2.7356 acc=0.1624 prec=0.1013 rec=0.1170 f1=0.0805


Epoch 9/10 - train:   0%|          | 1/283 [00:00<03:43,  1.26batch/s, loss=2.59]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 9/10 - val: 100%|██████████| 32/32 [00:15<00:00,  2.11batch/s, val_loss=2.74]


Epoch 9: train_loss=2.7622 val_loss=2.7404 acc=0.1653 prec=0.0761 rec=0.1174 f1=0.0785


Epoch 10/10 - train:  76%|███████▌  | 214/283 [01:53<00:39,  1.77batch/s, loss=2.76]c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 10/10 - val: 100%|██████████| 32/32 [00:14<00:00,  2.20batch/s, val_loss=2.81]


Epoch 10: train_loss=2.7560 val_loss=2.8065 acc=0.1524 prec=0.0967 rec=0.1075 f1=0.0739
